# <b><span style='color: #196f3d '>| PlantTraits2024:</span>  Ensemble Model: Image + Tabular [Inference]</b> 
<!-- Simple Tabular Data model based on [this notebook](https://www.kaggle.com/code/hdjojo/modified-planttraits2024-eda-training) -->
<!-- ### <b><span style='color: #196f3d '>Table of Contents</span></b> <a class='anchor' id='top'></a>
<div style=" background-color: #ecf0f1 ; padding: 13px 13px; border-radius: 8px; color: white">
<li><a href="#import_libraries">Import Libraries</a></li>
<li><a href="#Configuration">Configuration</a></li>
<li><a href="#load_data">Load Data</a></li>
<li><a href="#outlier_removal">Outlier Removal</a></li>
<li><a href="#pca">Principal Component Analysis</a></li>
<li><a href="#std">Auxillary Data</a></li>
<li><a href="#model">Model</a></li>
<li><a href="#train">Train Model</a></li>
<li><a href="#train">Score</a></li>
    
</div> -->

### <b><span style='color: #196f3d '>|</span> Import Libraries</b><a class='anchor' id='import_libraries'></a> [↑](#top) 

***

In [22]:
import os
import cv2
import torch
import glob
import time
import joblib
import torchmetrics
import timm
import warnings
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.optim as optim
import pytorch_lightning as pl
import albumentations as A
import imageio.v3 as imageio
import psutil
import seaborn as sns
import lightgbm as lgb
import pickle

from glob import glob
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from albumentations.pytorch import ToTensorV2
from torchmetrics.regression import R2Score
from sklearn.preprocessing import StandardScaler, Normalizer
from torch.optim.lr_scheduler import OneCycleLR
from torchvision.models import efficientnet
from sklearn.model_selection import StratifiedKFold, train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.decomposition import PCA

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

tqdm.pandas()

### <b><span style='color: #196f3d '>|</span> Configuration</b><a class='anchor' id='Configuration'></a> [↑](#top) 

***

In [23]:
class CFG:
    IMAGE_SIZE = 384
    BACKBONE = 'swin_large_patch4_window12_384.ms_in22k_ft_in1k'
    BATCH_SIZE = 10 
    NUM_CLASSES = 6 
    NUM_FOLDS = 5 
    FOLD = 0 # Which fold to set as validation set
    TARGET_COLUMNS = ['X4_mean', 'X11_mean', 'X18_mean', 'X50_mean', 'X26_mean', 'X3112_mean'] #don't change order 
    AUX_TARGET_COLUMNS = ['X4_sd', 'X11_sd', 'X18_sd', 'X50_sd', 'X26_sd', 'X3112_sd']
    SEED = 42  
    
# Set a seed using Pytorch Lightning
pl.seed_everything(CFG.SEED, workers=True)
# warnings.filterwarnings('ignore')

# Set base path
BASE_PATH = "/kaggle/input/planttraits2024"
DATAFRAME_PATH = "/kaggle/input/planttraits2024-dataframes"
BEST_MODEL_TABULAR = '/kaggle/input/v29-planttraits-rf-tabular-model-train-infer/tabular_model.pkl'
BEST_MODEL_IMAGE = '/kaggle/input/final-model-l1-no-log-scale/final_model.pth'

# <b><span style='color: #196f3d '>| Tabular Random Forest Inference Model </span><a class='anchor' id='load_data'></a> [↑](#top) 

### <b><span style='color: #196f3d '>|</span> Load Data</b><a class='anchor' id='load_data'></a> [↑](#top) 

In [24]:
tab_df = pd.read_csv(f'{BASE_PATH}/train.csv')
tab_test_df = pd.read_csv(f'{BASE_PATH}/test.csv')

#STORE TARGET COLLUMNS
CFG.FEATURE_COLS = tab_test_df.columns[1:].tolist() 

### <b><span style='color: #196f3d '>|</span> Outlier Removal</b><a class='anchor' id='preprocessing'></a> [↑](#top) 

In [25]:
#REMOVE OUTLIERS FROM MAIN AND AUXILLARY FEATURES FROM TRAINING DATA
for column in (CFG.TARGET_COLUMNS):
    upper_quantile = tab_df[column].quantile(0.98)
    tab_df = tab_df[(tab_df[column] < upper_quantile)]
    tab_df = tab_df[(tab_df[column] > 0)] #remove negative values 

#drop auxillary collumns
tab_df = tab_df.drop(columns=CFG.AUX_TARGET_COLUMNS)

### <b><span style='color: #196f3d '>|</span> Preprocessing </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [26]:
# LOG-SCALE FEATURES
train_tab_features = np.log1p(tab_df[CFG.FEATURE_COLS].values)
test_tab_features = np.log1p(tab_test_df[CFG.FEATURE_COLS].values)

# Normalize and store tabular features
FEATURE_SCALER = StandardScaler()
train_tab_features = FEATURE_SCALER.fit_transform(train_tab_features)
test_tab_features = FEATURE_SCALER.transform(test_tab_features)

# NORMALIZE TARGETS
TARGET_SCALER = StandardScaler()
train_targets = TARGET_SCALER.fit_transform(tab_df[CFG.TARGET_COLUMNS])

/tmp/ipykernel_34/202589056.py:2: RuntimeWarning: invalid value encountered in log1p
  train_tab_features = np.log1p(tab_df[CFG.FEATURE_COLS].values)
/tmp/ipykernel_34/202589056.py:3: RuntimeWarning: invalid value encountered in log1p
  test_tab_features = np.log1p(tab_test_df[CFG.FEATURE_COLS].values)


### <b><span style='color: #196f3d '>|</span> Inference </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [27]:
# Load in the model
with open(BEST_MODEL_TABULAR,'rb') as f:
    tabular_model = pickle.load(f)

# Get predictions
predictions_tab = tabular_model.predict(test_tab_features)

# Reverse the scaling
predictions_tab = TARGET_SCALER.inverse_transform(predictions_tab)

# Prepare submission dataframe
plot_tab_df = pd.DataFrame({'id': tab_test_df['id']}) #df to plot later
submission_tab_df = pd.DataFrame({'id': tab_test_df['id']})
target_cols = [x.replace("_mean", "") for x in CFG.TARGET_COLUMNS]

# Create submission file
plot_tab_df[CFG.TARGET_COLUMNS] = predictions_tab.tolist()
submission_tab_df[target_cols] = predictions_tab.tolist()
submission_tab_df

,id,X4,X11,X18,X50,X26,X3112
0,201238668,0.555062,10.234770,3.306154,1.550119,9.089028,715.500347
1,202310319,0.467770,17.757617,0.759625,1.486937,6.929776,1315.848532
2,202604412,0.409133,15.303750,1.251762,1.660520,5.049421,849.530634
3,201353439,0.450024,19.076018,0.599624,1.361695,4.087957,1052.247209
4,195351745,0.500283,9.163268,0.696625,1.601511,3.647701,550.645631
...,...,...,...,...,...,...,...
6540,195548469,0.678160,10.334983,1.967456,1.990803,14.602714,701.558794
6541,199261251,0.540943,16.253505,6.571631,1.472934,47.516186,4411.752671
6542,203031744,0.489114,20.040924,2.093139,1.266997,10.590708,1980.107724
6543,197736382,0.459265,19.201993,0.890531,1.370435,8.061711,1087.929783


# <b><span style='color: #196f3d '>| Image Vision Transformer Inference Model </span><a class='anchor' id='load_data'></a> [↑](#top) 

### <b><span style='color: #196f3d '>|</span> Load Data</b><a class='anchor' id='load_data'></a> [↑](#top) 

In [28]:
image_df = pd.read_pickle(f'{DATAFRAME_PATH}/df.pkl')
image_test_df = pd.read_pickle(f'{DATAFRAME_PATH}/test_df.pkl')

### <b><span style='color: #196f3d '>|</span> Outlier Removal</b><a class='anchor' id='preprocessing'></a> [↑](#top) 

In [29]:
# REMOVE OUTLIERS FROM MAIN AND TABULAR FEATURES FROM TRAINING DATA
for column in (CFG.TARGET_COLUMNS + CFG.AUX_TARGET_COLUMNS):
    upper_quantile = image_df[column].quantile(0.985)
    image_df = image_df[(image_df[column] < upper_quantile)]
    image_df = image_df[(image_df[column] > 0) | (image_df[column] == -1)] # remove negative values except -1

### <b><span style='color: #196f3d '>|</span> Validation </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [30]:
skf = StratifiedKFold(n_splits=CFG.NUM_FOLDS, shuffle=True, random_state=42)

# Create separate bin for each trait
for i, trait in enumerate(CFG.TARGET_COLUMNS):
    bin_edges = np.percentile(image_df[trait], np.linspace(0, 100, CFG.NUM_FOLDS + 1))
    image_df[f"bin_{i}"] = np.digitize(image_df[trait], bin_edges)
image_df["final_bin"] = image_df[[f"bin_{i}" for i in range(len(CFG.TARGET_COLUMNS))]].astype(str).agg("".join, axis=1)
image_df["fold"] = -1  # Initialize fold column

# Perform the stratified split using final bin
image_df = image_df.reset_index(drop=True)
for fold, (train_idx, valid_idx) in enumerate(skf.split(image_df, image_df["final_bin"])):
    image_df.loc[valid_idx, "fold"] = fold
    
# Drop the bin columns from the dataframe
bin_columns = [f"bin_{i}" for i in range(len(CFG.TARGET_COLUMNS))]
image_df = image_df.drop(columns= bin_columns +['final_bin'])
    
# Create a train and a validation dataframe based on splits
sample_df = image_df.copy()
train_df = sample_df[sample_df.fold != CFG.FOLD].reset_index(drop=True)
train_df.head()

/opt/conda/lib/python3.10/site-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


,id,WORLDCLIM_BIO1_annual_mean_temperature,WORLDCLIM_BIO12_annual_precipitation,WORLDCLIM_BIO13.BIO14_delta_precipitation_of_wettest_and_dryest_month,WORLDCLIM_BIO15_precipitation_seasonality,WORLDCLIM_BIO4_temperature_seasonality,WORLDCLIM_BIO7_temperature_annual_range,SOIL_bdod_0.5cm_mean_0.01_deg,SOIL_bdod_100.200cm_mean_0.01_deg,SOIL_bdod_15.30cm_mean_0.01_deg,...,X3112_mean,X4_sd,X11_sd,X18_sd,X26_sd,X50_sd,X3112_sd,image_path,image_bytes,fold
0,192027691,12.235703,374.466675,62.524445,72.256844,773.592041,33.277779,125,149,136,...,50.216034,0.008921,1.601473,0.025441,0.153608,0.279610,15.045054,/kaggle/input/planttraits2024/train_images/192...,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,1
1,195542235,17.270555,90.239998,10.351111,38.220940,859.193298,40.009777,124,144,138,...,574.098472,0.003102,0.258078,0.000866,0.034630,0.010165,11.004477,/kaggle/input/planttraits2024/train_images/195...,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,4
2,195728812,18.680834,1473.933350,163.100006,45.009758,381.053986,20.436666,120,131,125,...,1042.686546,0.011692,2.818356,0.110673,0.011334,0.229224,141.857187,/kaggle/input/planttraits2024/train_images/195...,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,3
3,195251545,0.673204,530.088867,50.857777,38.230709,1323.526855,45.891998,91,146,120,...,2386.467180,0.006157,1.128000,0.026996,0.553815,0.107092,87.146899,/kaggle/input/planttraits2024/train_images/195...,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,2
4,195733955,12.062123,1982.033325,320.138092,74.343796,318.258270,17.557619,101,142,120,...,363.364702,0.002750,0.352804,0.041009,7.854885,0.009494,14.671018,/kaggle/input/planttraits2024/train_images/195...,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...,1


### <b><span style='color: #196f3d '>|</span> Preprocessing </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [31]:
# STORE MAIN FEATURES
train_image_features = train_df['image_bytes'].values
test_image_features  = image_test_df['image_bytes'].values

# STORE AND NORMALIZE MAIN TARGETS
TARGET_SCALER = StandardScaler()
train_targets = TARGET_SCALER.fit_transform(train_df[CFG.TARGET_COLUMNS].values)

print('N_TRAIN_SAMPLES:', len(train_df),'N_TEST_SAMPLES:', len(image_test_df))

N_TRAIN_SAMPLES: 36872 N_TEST_SAMPLES: 6545


### <b><span style='color: #196f3d '>|</span> Augmentations </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [32]:
IMAGENET_MEAN = [0.485, 0.456, 0.406] #statistics of image_net photos, used to normalize images
IMAGENET_STD = [0.229, 0.224, 0.225] 

TEST_TRANSFORMS = A.Compose([
        A.Resize(CFG.IMAGE_SIZE, CFG.IMAGE_SIZE),
        A.ToFloat(),#float32
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=1), #normalization to match the statistics used to pre-train the EfficientNet model
        ToTensorV2(),
    ])

### <b><span style='color: #196f3d '>|</span> Dataset </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [33]:
class PlantDataset(Dataset):
    def __init__(self, image_bytes, tab_features = None, targets = None, aux_targets = None, transforms=None):
        self.image_bytes = image_bytes
        self.tab_features = tab_features
        self.targets = targets
        self.aux_targets = aux_targets
        self.transforms = transforms

    def __len__(self):
        return len(self.image_bytes)

    def __getitem__(self, idx):
        #read the image data directly from its JPEG-encoded byte representation
        image = self.transforms(image=imageio.imread(self.image_bytes[idx]))['image'] #Access the the modified image array with shape (size,size,3) within the Albumentations dictionary 
        
        if self.tab_features is not None:
            tab_feature = self.tab_features[idx]
        else:
            tab_feature = None
            
        target = torch.tensor(self.targets[idx])
        aux_target = torch.tensor(self.aux_targets[idx], dtype=torch.float32)
        
        if tab_feature is not None:
            return {'images': image, 'tab_features': tab_feature}, (target, aux_target)
        else:
            return {'images': image}, (target, aux_target)

### <b><span style='color: #196f3d '>|</span> Dataloader </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [34]:
test_dataset = PlantDataset(test_image_features, tab_features = None, targets = image_test_df['id'].values, aux_targets = image_test_df['id'].values, transforms = TEST_TRANSFORMS)

test_dataloader = DataLoader(
        test_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=1,
)

### <b><span style='color: #196f3d '>|</span> Image Model </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [35]:
class ImageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
                CFG.BACKBONE,
                num_classes=6,
                pretrained=True)
        
    def forward(self, inputs):
        return self.backbone(inputs)

### <b><span style='color: #196f3d '>|</span> Inference </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [36]:
# Instantiate the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImageModel()
model.to(device)

#Load the best model
model.load_state_dict(torch.load(BEST_MODEL_IMAGE))

# List to store predictions
SUBMISSION_ROWS = []

# Iterate over batches in the test data loader
model.eval()
for batch_idx, (inputs_dict, test_ids) in enumerate(tqdm(test_dataloader, desc='Testing')):
    # Extract images and features from the inputs_dict
    inputs_images = inputs_dict['images'].to(device, dtype=torch.float32)  

    # Forward pass them through the model
    with torch.no_grad():
        predictions_image = model(inputs_images).cpu().numpy()

    #Reverse the scaling done
    predictions_image = TARGET_SCALER.inverse_transform(predictions_image)

    # Append predictions to the list
    SUBMISSION_ROWS.append(predictions_image)

# Concatenate predictions for all batches
SUBMISSION_ROWS = np.concatenate(SUBMISSION_ROWS, axis=0)

Testing:   0%|          | 0/655 [00:00<?, ?it/s]

In [37]:
submission_image_df = image_test_df[["id"]].copy()
target_cols = [x.replace("_mean","") for x in CFG.TARGET_COLUMNS]
submission_image_df[target_cols] = SUBMISSION_ROWS.tolist()
submission_image_df

,id,X4,X11,X18,X50,X26,X3112
0,201238668,0.570943,5.619057,3.462222,1.852579,2.750393,423.475250
1,202310319,0.570784,19.019224,0.408676,0.971274,1.156599,1508.598877
2,202604412,0.624311,15.798512,2.193650,1.367478,31.791870,1106.501221
3,201353439,0.548734,21.819403,0.276587,1.235743,5.323733,1337.305664
4,195351745,0.517366,11.333061,0.620439,1.532701,3.535837,272.194550
...,...,...,...,...,...,...,...
6540,195548469,0.715047,8.531357,1.677058,2.492206,5.978006,633.144897
6541,199261251,0.521799,19.801516,5.494149,1.351298,27.010653,3855.504639
6542,203031744,0.431643,27.193092,0.195120,1.112578,5.920363,2168.023682
6543,197736382,0.480701,22.911774,-0.223566,1.416687,1.104580,239.652679


# <b><span style='color: #196f3d '>|</span> Ensemble Inference </b><a class='anchor' id='load_data'></a> [↑](#top) 

In [42]:
submission_df = 0.19*submission_tab_df + 0.81*submission_image_df
submission_df["id"] = image_test_df[["id"]].copy()
submission_df.to_csv('submission.csv', index=False)
print("Submitted!")
submission_df

Submitted!


,id,X4,X11,X18,X50,X26,X3112
0,201238668,0.567767,6.542199,3.431008,1.792087,4.018120,481.880270
1,202310319,0.550181,18.766903,0.478866,1.074406,2.311234,1470.048808
2,202604412,0.581275,15.699560,2.005272,1.426087,26.443380,1055.107103
3,201353439,0.528992,21.270726,0.341194,1.260933,5.076578,1280.293973
4,195351745,0.513950,10.899103,0.635676,1.546463,3.558210,327.884766
...,...,...,...,...,...,...,...
6540,195548469,0.707669,8.892082,1.735137,2.391925,7.702947,646.827677
6541,199261251,0.525628,19.091913,5.709645,1.375625,31.111759,3966.754245
6542,203031744,0.443137,25.762659,0.574724,1.143462,6.854432,2130.440490
6543,197736382,0.476413,22.169818,-0.000746,1.407436,2.496006,409.308100
